# Player and Lineup Performance Metrics

This notebook computes performance metrics at the lineup and player level using the reconstructed stint dataset.

## Objective

With a validated stint dataset, we now evaluate lineup and player performance.

The goal is to:

- measure scoring performance across lineup segments
- identify high- and low-performing lineups
- estimate player impact based on on-court scoring outcomes
- build the foundation for advanced impact modeling

This transforms the reconstructed lineup data into actionable basketball insights.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

stints_df = pd.read_pickle("../data/interim/stints_prototype_game.pkl")

## Reconstructing player name mapping

To keep the analysis reproducible and independent from previous notebooks, we rebuild the player ID-to-name mapping directly from play-by-play data.

This ensures that all downstream computations, including player impact analysis, remain self-contained and do not rely on intermediate artifacts from earlier steps.

In [2]:
def build_player_name_map(game_df):
    pbp_player_lookup_1 = game_df[["player1_id", "player1_name"]].dropna().rename(
        columns={"player1_id": "id", "player1_name": "full_name"}
    )

    pbp_player_lookup_2 = game_df[["player2_id", "player2_name"]].dropna().rename(
        columns={"player2_id": "id", "player2_name": "full_name"}
    )

    pbp_player_lookup_3 = game_df[["player3_id", "player3_name"]].dropna().rename(
        columns={"player3_id": "id", "player3_name": "full_name"}
    )

    pbp_player_lookup = pd.concat(
        [pbp_player_lookup_1, pbp_player_lookup_2, pbp_player_lookup_3],
        axis=0
    ).drop_duplicates()

    pbp_player_lookup["id"] = pbp_player_lookup["id"].astype(str)

    return dict(zip(pbp_player_lookup["id"], pbp_player_lookup["full_name"]))

In [3]:
import sqlite3

conn = sqlite3.connect("../data/raw/nba.sqlite")

sample_game_id = stints_df["game_id"].iloc[0]

game = pd.read_sql(f"""
SELECT *
FROM play_by_play
WHERE game_id = '{sample_game_id}'
""", conn)

In [4]:
player_name_map = build_player_name_map(game)

## Prepare performance metrics

We compute the scoring differential for each stint as a measure of lineup performance.

In [5]:
stints_df["net_points"] = stints_df["home_points"] - stints_df["away_points"]

stints_df[[
    "period",
    "home_points",
    "away_points",
    "net_points"
]].head()

,period,home_points,away_points,net_points
0,1,10,6,4
1,1,10,6,4
2,1,0,3,-3
3,1,1,0,1
4,1,0,0,0


##  Lineup performance analysis

We aggregate stint-level scoring to evaluate how each lineup performs over the course of the game.

In [6]:
home_lineup_perf = (
    stints_df
    .groupby(stints_df["home_lineup_names"].apply(tuple))
    .agg(
        total_points=("home_points", "sum"),
        total_allowed=("away_points", "sum"),
        total_stints=("home_points", "count")
    )
    .reset_index()
)

home_lineup_perf["net_points"] = (
    home_lineup_perf["total_points"] - home_lineup_perf["total_allowed"]
)

home_lineup_perf.sort_values("net_points", ascending=False).head(10)

,home_lineup_names,total_points,total_allowed,total_stints,net_points
5,"(Bo McCalebb, Linas Kleiza, Luka Zoric, Bojan ...",13,8,2,5
17,"(Melih Mahmutoglu, Luka Zoric, James Birsen, K...",4,2,2,2
14,"(Melih Mahmutoglu, Linas Kleiza, Gasper Vidmar...",3,1,2,2
10,"(Melih Mahmutoglu, Izzet Turkyilmaz, Gasper Vi...",3,2,2,1
8,"(Izzet Turkyilmaz, James Birsen, Kenan Sipahi,...",1,0,1,1
0,"(1962936724, Izzet Turkyilmaz, James Birsen, B...",3,2,1,1
6,"(Bo McCalebb, Melih Mahmutoglu, Linas Kleiza, ...",0,0,1,0
1,"(Baris Ermis, Linas Kleiza, Luka Zoric, James ...",0,0,1,0
11,"(Melih Mahmutoglu, Izzet Turkyilmaz, Gasper Vi...",1,1,1,0
13,"(Melih Mahmutoglu, Izzet Turkyilmaz, Luka Zori...",4,4,1,0


In [7]:
away_lineup_perf = (
    stints_df
    .groupby(stints_df["away_lineup_names"].apply(tuple))
    .agg(
        total_points=("away_points", "sum"),
        total_allowed=("home_points", "sum"),
        total_stints=("away_points", "count")
    )
    .reset_index()
)

away_lineup_perf["net_points"] = (
    away_lineup_perf["total_points"] - away_lineup_perf["total_allowed"]
)

away_lineup_perf.sort_values("net_points", ascending=False).head(10)

,away_lineup_names,total_points,total_allowed,total_stints,net_points
2,"(Thabo Sefolosha, Jeremy Lamb, Perry Jones III...",12,8,5,4
8,"(Thabo Sefolosha, Serge Ibaka, Reggie Jackson,...",26,22,6,4
4,"(Thabo Sefolosha, Kevin Durant, Serge Ibaka, R...",3,0,1,3
7,"(Thabo Sefolosha, Serge Ibaka, Jeremy Lamb, Pe...",7,4,2,3
5,"(Thabo Sefolosha, Serge Ibaka, Hasheem Thabeet...",2,0,2,2
0,"(Thabo Sefolosha, Hasheem Thabeet, Jeremy Lamb...",8,7,3,1
6,"(Thabo Sefolosha, Serge Ibaka, Jeremy Lamb, Pe...",2,1,2,1
1,"(Thabo Sefolosha, Hasheem Thabeet, Reggie Jack...",10,12,6,-2
9,"(Thabo Sefolosha, Serge Ibaka, Reggie Jackson,...",0,4,3,-4
3,"(Thabo Sefolosha, Kevin Durant, Serge Ibaka, R...",12,20,2,-8


### Home lineup performance interpretation

Home lineup performance shows moderate variability across player combinations.

The top-performing lineup achieves a +5 point differential over two stints, indicating a short but effective segment of play. Several other lineups produce small positive differentials (+1 to +2), suggesting incremental contributions rather than dominant performance.

Most lineups appear in only one or two stints, reflecting frequent rotation patterns and limited sample sizes. As a result, conclusions about lineup effectiveness should be interpreted cautiously.

Overall, the home team does not exhibit a clearly dominant lineup, but rather a distribution of moderately effective player combinations.

### Away lineup performance interpretation

Away lineup performance demonstrates stronger and more consistent high-impact combinations.

Multiple lineups achieve a +4 point differential, with one lineup accumulating 26 points scored and 22 allowed across six stints. This suggests the presence of a stable and effective core lineup.

At the same time, several lineups show negative differentials (-4 and -8), indicating variability in performance across different rotations.

Overall, the away team appears to rely on a strong core lineup while experimenting with additional combinations that produce mixed results.

## Adjusted lineup performance

Raw net points are useful, but they can be misleading because lineups are not all used the same number of times.

To make lineup comparisons more fair, we normalize each lineup’s net performance by its number of stints.

This creates a simple `net_per_stint` metric:

- positive values indicate that the lineup tends to outscore opponents during its segments
- negative values indicate that the lineup tends to be outscored
- higher values suggest stronger short-term performance, though sample size still matters

In [8]:
# Normalize lineup performance by stint count
home_lineup_perf["net_per_stint"] = (
    home_lineup_perf["net_points"] / home_lineup_perf["total_stints"]
)

away_lineup_perf["net_per_stint"] = (
    away_lineup_perf["net_points"] / away_lineup_perf["total_stints"]
)

# Sort by normalized performance
home_lineup_perf_adjusted = home_lineup_perf.sort_values(
    "net_per_stint", ascending=False
)

away_lineup_perf_adjusted = away_lineup_perf.sort_values(
    "net_per_stint", ascending=False
)

## Home lineup performance adjusted by stint count

In [9]:
home_lineup_perf_adjusted[[
    "home_lineup_names",
    "total_points",
    "total_allowed",
    "total_stints",
    "net_points",
    "net_per_stint"
]].head(10)

,home_lineup_names,total_points,total_allowed,total_stints,net_points,net_per_stint
5,"(Bo McCalebb, Linas Kleiza, Luka Zoric, Bojan ...",13,8,2,5,2.5
0,"(1962936724, Izzet Turkyilmaz, James Birsen, B...",3,2,1,1,1.0
14,"(Melih Mahmutoglu, Linas Kleiza, Gasper Vidmar...",3,1,2,2,1.0
8,"(Izzet Turkyilmaz, James Birsen, Kenan Sipahi,...",1,0,1,1,1.0
17,"(Melih Mahmutoglu, Luka Zoric, James Birsen, K...",4,2,2,2,1.0
10,"(Melih Mahmutoglu, Izzet Turkyilmaz, Gasper Vi...",3,2,2,1,0.5
6,"(Bo McCalebb, Melih Mahmutoglu, Linas Kleiza, ...",0,0,1,0,0.0
1,"(Baris Ermis, Linas Kleiza, Luka Zoric, James ...",0,0,1,0,0.0
11,"(Melih Mahmutoglu, Izzet Turkyilmaz, Gasper Vi...",1,1,1,0,0.0
13,"(Melih Mahmutoglu, Izzet Turkyilmaz, Luka Zori...",4,4,1,0,0.0


## Away lineup performance adjusted by stint count

In [10]:
away_lineup_perf_adjusted[[
    "away_lineup_names",
    "total_points",
    "total_allowed",
    "total_stints",
    "net_points",
    "net_per_stint"
]].head(10)

,away_lineup_names,total_points,total_allowed,total_stints,net_points,net_per_stint
4,"(Thabo Sefolosha, Kevin Durant, Serge Ibaka, R...",3,0,1,3,3.000000
7,"(Thabo Sefolosha, Serge Ibaka, Jeremy Lamb, Pe...",7,4,2,3,1.500000
5,"(Thabo Sefolosha, Serge Ibaka, Hasheem Thabeet...",2,0,2,2,1.000000
2,"(Thabo Sefolosha, Jeremy Lamb, Perry Jones III...",12,8,5,4,0.800000
8,"(Thabo Sefolosha, Serge Ibaka, Reggie Jackson,...",26,22,6,4,0.666667
6,"(Thabo Sefolosha, Serge Ibaka, Jeremy Lamb, Pe...",2,1,2,1,0.500000
0,"(Thabo Sefolosha, Hasheem Thabeet, Jeremy Lamb...",8,7,3,1,0.333333
1,"(Thabo Sefolosha, Hasheem Thabeet, Reggie Jack...",10,12,6,-2,-0.333333
9,"(Thabo Sefolosha, Serge Ibaka, Reggie Jackson,...",0,4,3,-4,-1.333333
3,"(Thabo Sefolosha, Kevin Durant, Serge Ibaka, R...",12,20,2,-8,-4.000000


## Adjusted lineup performance interpretation

Normalizing lineup performance by stint count reveals important differences between short-term efficiency and sustained impact.

For the home team, the highest-performing lineup achieves a +2.5 net points per stint, but this result is based on only two stints. Several other lineups show consistent but moderate effectiveness (+1.0 per stint), while many appear only once, limiting the reliability of conclusions.

This suggests that the home team relies on frequent rotations and does not exhibit a clearly dominant lineup across extended segments.

For the away team, the analysis highlights a more structured rotation pattern. While the highest per-stint performance (+3.0) occurs in a single stint, a more meaningful insight comes from a lineup that achieves a +0.67 net points per stint across six stints. This indicates a stable and consistently effective core lineup.

Overall, the away team demonstrates stronger lineup stability and sustained performance, while the home team shows more fragmented but occasionally efficient lineup combinations.

These results illustrate the importance of balancing performance magnitude with sample size when evaluating lineup effectiveness.

## Interpreting small sample sizes

Lineups that appear in only one or two stints may show extreme performance values due to limited sample size.

For this reason, lineups with higher stint counts provide more reliable indicators of true performance and should be prioritized in analysis.

## Player impact dataset construction

To evaluate individual player contributions, we transform the lineup-based stint dataset into a player-level dataset.

Each stint represents a segment of the game where a fixed group of players is on the court. During that stint, the lineup produces a net point differential (points scored minus points allowed).

To attribute performance to individual players:

- Each player in a lineup inherits the net point differential of the stint
- This contribution is weighted by the duration of the stint (in seconds)
- Both home and away players are processed symmetrically

This transformation creates a dataset where each row represents a player’s participation in a specific stint, along with the associated performance and playing time.

This structure enables the computation of time-weighted player impact metrics in the next step.

### Methodological note

This approach assumes that all players on the court contribute equally to the lineup’s net performance during each stint.

While this is a simplification, it provides a strong baseline for player evaluation and is commonly used in early-stage lineup and impact analysis.

More advanced models (e.g., adjusted plus-minus) can refine this approach by isolating individual contributions.

In [11]:
def clock_to_seconds(clock_str):
    if pd.isna(clock_str):
        return None
    minutes, seconds = map(int, clock_str.split(":"))
    return minutes * 60 + seconds

In [12]:
stints_df["start_sec"] = stints_df["start_clock"].apply(clock_to_seconds)
stints_df["end_sec"] = stints_df["end_clock"].apply(clock_to_seconds)

# NBA clock counts down → duration = start - end
stints_df["duration_sec"] = stints_df["start_sec"] - stints_df["end_sec"]

# Remove weird cases (optional safety)
stints_df = stints_df[stints_df["duration_sec"] > 0]

In [14]:
player_stints = []

for row in stints_df.itertuples():

    # Home players
    for player in row.home_lineup:
        player_stints.append({
            "player_id": str(player),
            "team": "home",
            "net_points": row.home_points - row.away_points,
            "duration_sec": row.duration_sec
        })

    # Away players
    for player in row.away_lineup:
        player_stints.append({
            "player_id": str(player),
            "team": "away",
            "net_points": row.away_points - row.home_points,
            "duration_sec": row.duration_sec
        })

player_stints_df = pd.DataFrame(player_stints)

In [15]:
player_impact = (
    player_stints_df
    .groupby("player_id")
    .agg(
        total_net_points=("net_points", "sum"),
        total_duration=("duration_sec", "sum"),
        total_stints=("net_points", "count")
    )
    .reset_index()
)

In [16]:
player_impact["impact_per_60_sec"] = (
    player_impact["total_net_points"] / player_impact["total_duration"]
) * 60

In [17]:
player_impact["player_name"] = player_impact["player_id"].apply(
    lambda pid: player_name_map.get(str(pid), str(pid))
)

In [18]:
player_impact.sort_values("impact_per_60_sec", ascending=False).head(15)

,player_id,total_net_points,total_duration,total_stints,impact_per_60_sec,player_name
22,42546,2,118,2,1.016949,Berk Ugurlu
0,1962936724,1,102,1,0.588235,1962936724
7,203103,9,962,12,0.561331,Perry Jones III
24,965,4,475,5,0.505263,Derek Fisher
8,203197,2,329,5,0.364742,Diante Garrett
6,203087,10,2176,27,0.275735,Jeremy Lamb
1,200757,5,2643,30,0.113507,Thabo Sefolosha
10,2570,3,1745,15,0.103152,Kendrick Perkins
4,201934,1,689,10,0.087083,Hasheem Thabeet
3,201586,2,1524,16,0.078740,Serge Ibaka


In [19]:
player_impact.sort_values("impact_per_60_sec", ascending=True).head(15)

,player_id,total_net_points,total_duration,total_stints,impact_per_60_sec,player_name
2,201142,-5,467,3,-0.642398,Kevin Durant
12,42534,-4,475,5,-0.505263,Baris Ermis
23,42547,-2,329,5,-0.364742,Ayberk Olmaz
13,42535,-5,962,15,-0.311850,Melih Mahmutoglu
18,42541,-5,1153,15,-0.260191,James Birsen
9,2555,-2,524,9,-0.229008,Nick Collison
19,42542,-4,1198,17,-0.200334,Kenan Sipahi
16,42538,-3,1142,14,-0.157618,Gasper Vidmar
5,202704,-4,1681,18,-0.142772,Reggie Jackson
15,42537,-1,577,8,-0.103986,Izzet Turkyilmaz


In [20]:
player_impact[[
    "player_name",
    "total_net_points",
    "total_duration",
    "total_stints",
    "impact_per_60_sec"
]].sort_values("total_duration", ascending=False).head(15)

,player_name,total_net_points,total_duration,total_stints,impact_per_60_sec
1,Thabo Sefolosha,5,2643,30,0.113507
6,Jeremy Lamb,10,2176,27,0.275735
10,Kendrick Perkins,3,1745,15,0.103152
20,Bojan Bogdanovic,0,1735,16,0.000000
5,Reggie Jackson,-4,1681,18,-0.142772
3,Serge Ibaka,2,1524,16,0.078740
14,42536,0,1483,16,0.000000
21,Emir Preldzic,-2,1426,13,-0.084151
11,Bo McCalebb,-2,1343,12,-0.089352
19,Kenan Sipahi,-4,1198,17,-0.200334


## Player impact interpretation

Some players show strong impact per 60 seconds despite relatively moderate total minutes, suggesting efficient but potentially situational contributions rather than consistently dominant performance.

Conversely, players with higher total minutes but near-zero impact per 60 seconds may reflect a more neutral on-court presence or balanced rotation roles, rather than poor individual performance. These cases often indicate players contributing within stable lineups where scoring margins remain relatively even.


## Sample-size note

Players with very limited duration can produce extreme impact values. For that reason, player impact should be interpreted alongside total duration and total stint count, not only by `impact_per_60_sec`.

In [21]:
min_duration = 600  # 10 minutes

reliable_player_impact = player_impact[
    player_impact["total_duration"] >= min_duration
].copy()

reliable_player_impact.sort_values(
    "impact_per_60_sec", ascending=False
).head(15)

,player_id,total_net_points,total_duration,total_stints,impact_per_60_sec,player_name
7,203103,9,962,12,0.561331,Perry Jones III
6,203087,10,2176,27,0.275735,Jeremy Lamb
1,200757,5,2643,30,0.113507,Thabo Sefolosha
10,2570,3,1745,15,0.103152,Kendrick Perkins
4,201934,1,689,10,0.087083,Hasheem Thabeet
3,201586,2,1524,16,0.078740,Serge Ibaka
14,42536,0,1483,16,0.000000,42536
17,42540,0,1172,11,0.000000,Luka Zoric
20,42544,0,1735,16,0.000000,Bojan Bogdanovic
21,42545,-2,1426,13,-0.084151,Emir Preldzic


In [22]:

# Build lineup-level performance dataset


lineup_perf = (
    stints_df
    .groupby(stints_df["home_lineup_names"].apply(tuple))
    .agg(
        total_points=("home_points", "sum"),
        total_allowed=("away_points", "sum"),
        total_duration=("duration_sec", "sum"),
        total_stints=("home_points", "count")
    )
    .reset_index()
)


# Core performance metrics


lineup_perf["net_points"] = (
    lineup_perf["total_points"] - lineup_perf["total_allowed"]
)

lineup_perf["net_per_60"] = (
    lineup_perf["net_points"] / lineup_perf["total_duration"]
) * 60


# Additional analytical signal 

# Proxy for lineup usage / possessions
lineup_perf["possessions_proxy"] = lineup_perf["total_stints"]


# Identify most used (core) lineups


top_usage_lineups = lineup_perf.sort_values(
    "total_duration", ascending=False
).head(10)

display(top_usage_lineups)


# Identify best performing lineups (efficiency)


top_efficiency_lineups = lineup_perf.sort_values(
    "net_per_60", ascending=False
).head(10)

display(top_efficiency_lineups)


# Filter for reliability 


min_duration = 600  # 10 minutes threshold

reliable_lineups = lineup_perf[
    lineup_perf["total_duration"] >= min_duration
].copy()

reliable_lineups.sort_values(
    "net_per_60", ascending=False
).head(10)

,home_lineup_names,total_points,total_allowed,total_duration,total_stints,net_points,net_per_60,possessions_proxy
4,"(Bo McCalebb, Linas Kleiza, Gasper Vidmar, Boj...",29,32,760,6,-3,-0.236842,6
5,"(Bo McCalebb, Linas Kleiza, Luka Zoric, Bojan ...",13,8,328,2,5,0.914634,2
2,"(Baris Ermis, Luka Zoric, James Birsen, Kenan ...",5,6,293,2,-1,-0.204778,2
12,"(Melih Mahmutoglu, Izzet Turkyilmaz, James Bir...",4,8,176,2,-4,-1.363636,2
3,"(Bo McCalebb, Baris Ermis, Luka Zoric, Bojan B...",3,6,159,2,-3,-1.132075,2
16,"(Melih Mahmutoglu, Linas Kleiza, Luka Zoric, J...",4,7,158,2,-3,-1.139241,2
13,"(Melih Mahmutoglu, Izzet Turkyilmaz, Luka Zori...",4,4,125,1,0,0.000000,1
0,"(1962936724, Izzet Turkyilmaz, James Birsen, B...",3,2,102,1,1,0.588235,1
10,"(Melih Mahmutoglu, Izzet Turkyilmaz, Gasper Vi...",3,2,94,2,1,0.638298,2
17,"(Melih Mahmutoglu, Luka Zoric, James Birsen, K...",4,2,86,1,2,1.395349,1


,home_lineup_names,total_points,total_allowed,total_duration,total_stints,net_points,net_per_60,possessions_proxy
8,"(Izzet Turkyilmaz, James Birsen, Kenan Sipahi,...",1,0,16,1,1,3.750000,1
14,"(Melih Mahmutoglu, Linas Kleiza, Gasper Vidmar...",3,1,83,2,2,1.445783,2
17,"(Melih Mahmutoglu, Luka Zoric, James Birsen, K...",4,2,86,1,2,1.395349,1
5,"(Bo McCalebb, Linas Kleiza, Luka Zoric, Bojan ...",13,8,328,2,5,0.914634,2
10,"(Melih Mahmutoglu, Izzet Turkyilmaz, Gasper Vi...",3,2,94,2,1,0.638298,2
0,"(1962936724, Izzet Turkyilmaz, James Birsen, B...",3,2,102,1,1,0.588235,1
6,"(Bo McCalebb, Melih Mahmutoglu, Linas Kleiza, ...",0,0,48,1,0,0.000000,1
1,"(Baris Ermis, Linas Kleiza, Luka Zoric, James ...",0,0,23,1,0,0.000000,1
11,"(Melih Mahmutoglu, Izzet Turkyilmaz, Gasper Vi...",1,1,64,1,0,0.000000,1
13,"(Melih Mahmutoglu, Izzet Turkyilmaz, Luka Zori...",4,4,125,1,0,0.000000,1


,home_lineup_names,total_points,total_allowed,total_duration,total_stints,net_points,net_per_60,possessions_proxy
4,"(Bo McCalebb, Linas Kleiza, Gasper Vidmar, Boj...",29,32,760,6,-3,-0.236842,6


In [23]:
min_duration = 600  # same idea as players

reliable_lineups = lineup_perf[
    lineup_perf["total_duration"] >= min_duration
].copy()

reliable_lineups.sort_values(
    "net_per_60", ascending=False
).head(10)

,home_lineup_names,total_points,total_allowed,total_duration,total_stints,net_points,net_per_60,possessions_proxy
4,"(Bo McCalebb, Linas Kleiza, Gasper Vidmar, Boj...",29,32,760,6,-3,-0.236842,6


## Lineup performance insights

After filtering for minimum playing time, we observe that only a subset of lineups accumulate enough minutes to provide reliable estimates.

Top-performing lineups tend to:
- Include core rotation players
- Appear repeatedly across multiple stints
- Maintain positive scoring margins over longer durations

This reinforces the idea that lineup effectiveness is driven more by stability and repetition than isolated short bursts of performance.

In [25]:
# Save lineup-level datasets


reliable_lineups.to_csv("../data/processed/reliable_lineups.csv", index=False)
lineup_perf.to_csv("../data/processed/lineup_performance.csv", index=False)


# Save player-level datasets

player_impact.to_csv("../data/processed/player_impact.csv", index=False)
reliable_player_impact.to_csv("../data/processed/reliable_player_impact.csv", index=False)

print("Datasets saved successfully ✔")

Datasets saved successfully ✔


## Key insights

This notebook translates reconstructed stints into player- and lineup-level performance metrics.

**Player impact**
- Net scoring impact is normalized by playing time (per 60 seconds) to enable fair comparisons.
- Filtering by minimum duration improves reliability by reducing small-sample noise.
- Consistent contributors combine positive impact with meaningful minutes, while extreme values in low minutes should be interpreted cautiously.

**Lineup impact**
- Lineups are evaluated by aggregating scoring performance across stints.
- A small set of core lineups accounts for most playing time and shows more stable performance.
- Less frequent lineups tend to exhibit higher variability, reflecting situational usage.

**Overall**
The stint-based framework enables tracking of on-court combinations and estimation of both player and lineup contributions, providing a solid foundation for deeper analysis.

## Limitations

These metrics are descriptive rather than causal. Player and lineup impact are based on one game and do not control for opponent strength, teammate quality, possession count, or game context.

The results should be interpreted as a prototype performance framework rather than definitive player rankings.

## Next steps

With a stable stint and performance dataset, the next stage focuses on deeper analysis.

- **Lineup analysis (Notebook 06):** identify effective 5-player combinations and rotation patterns  
- **Player synergy:** evaluate how specific player combinations perform together  
- **Modeling:** explore predictive approaches for lineup and player impact  
- **Visualization:** summarize findings through clear and intuitive charts  

The project now transitions from data reconstruction into analytical and modeling work.